<h2><b>计算机高等教育通用教材</b></h2>
<h2>机器学习 Machine learning</h2>
<hr>
<h5>第一部分：监督学习 supervised learning</h5>
<h5>第二章：回归 regression</h5>
<hr>
<h3><b>实验二：基于线性回归（Linear Regression）算法的预测实操</b></h3>
<hr>
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html'>查看LinearRegression源代码(sklearn)</a><br>
<br>

> **适合人群** ：只要你学过了我们的《实验一：KNN分类》，理解了基础的机器学习流程（数据探查、切分、标准化、模型评估），你就能轻松拿下本章！本实验将带你预测**连续的数值**（比如房价、股票、病情指数），而不是分类（比如判断花朵种类）。
<hr>

#### 第0步：测试python与虚拟环境

In [ ]:
print("Hello Regression World!")
import pip
print("Pip version:", pip.__version__)

<hr><hr>

#### 第一步：import库 & 导入数据
<hr>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# 导入 sklearn 相关的库
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression as SklearnLinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor

In [ ]:
# 这里我们使用 sklearn 自带的【糖尿病数据集】进行示范
# 数据集包含了 442 个糖尿病患者的生理数据（特征），以及一年后病情的发展阶段指数（标签/目标值）

# load_diabetes 默认会对数据进行中心化和缩放，为了和最原始的医疗记录保持一致，方便我们直观理解数据，
# 我们使用 as_frame=True 并且尝试还原原始数据的粗略样貌（或者直接使用未缩放版本如果版本支持）。
diabetes = load_diabetes(as_frame=True, scaled=False)
df = diabetes.data

# 原数据集特征名是英文缩写，我们给数据集各列起个中文名字，降低理解负担
df.columns = ['年龄', '性别', '身体质量指数(BMI)', '血压', '总胆固醇', '低密度脂蛋白', '高密度脂蛋白', '甲状腺素', '血清甘油三酯', '血糖'] 
df['发展阶段指数(病期)'] = diabetes.target # 这是我们要预测的 y（连续数值）

print(f'数据集大小: {df.shape}')
df.head()
# 使用 head() 函数，显示前 5 行。你会看到年龄是类似 59 这样的数字，BMI 是 32.1，这就是真实世界的数据样貌。

<hr><hr>

#### 第二步：查看数据的基本信息（数据探查 EDA）
<hr>

In [ ]:
df.info()
# 看看数据类型，都是 float，没有缺失值 (Non-Null)，非常干净。真实世界的数据往往没这么干净，需要我们手动填补缺失值。

In [ ]:
df.describe()
# describe() 是个好东西。它可以看平均值(mean)、标准差(std)和百分位数(25%, 50%, 75%)。
# 百分位数知识专栏：将一组数据从小到大排序并计算累计百分位，某个百分位对应的数据值即为该百分位的百分位数。
# 比如 50% 分位数（中位数）和 mean（平均值）如果差得特别远，说明数据里可能有极端的“异常值”（比如马云走进了一个普通人的酒吧，酒吧平均收入被拉高了，但中位数还是普通人的收入）。

In [ ]:
# 数据分段（连续数据离散化）
# 在数据分析中，面对“年龄”这种连续数字，我们有时候想把它变成类别（比如 40岁以下，40-50岁...），方便统计。
# Pandas 提供了 cut() 函数来实现数据分段。

bins = [0, 40, 50, 60, 70, 100]
labels = ['40岁以下', '40~49岁', '50~59岁', '60~69岁', '70岁以上']
df['年龄段'] = pd.cut(df['年龄'], bins=bins, labels=labels)

age_count = df['年龄段'].value_counts().sort_index()
print("各年龄段患者人数分布：\n", age_count)

In [ ]:
# 看看性别分布 (在原始数据中，1 代表男性，2 代表女性)
gender_count = df['性别'].value_counts()
gender_count.index = ['男', '女']
print("\n性别特征统计：\n", gender_count)

<hr><hr>

#### 第三步：数据可视化
<hr>

In [ ]:
import matplotlib.font_manager as fm

# 配置中文字体，防止图表中的中文变成小方块
zh_fonts = [f.name for f in fm.fontManager.ttflist 
            if any(kw in f.name for kw in ['Hei', 'Song', 'CJK', 'Chinese', 'SC', 'TC', 'Gothic', 'SimHei'])]
if zh_fonts:
    plt.rcParams['font.family'] = zh_fonts[0]
plt.rcParams['axes.unicode_minus'] = False # 正常显示负号

In [ ]:
# 1. 箱型图：查看是否存在极端值
# 箱子的上下边缘是 75% 和 25% 分位数，中间的线是中位数。外面的圆圈是离群点（异常值）。
fig = plt.figure(figsize=(10, 4))
ax1 = fig.add_subplot(121)
ax1.boxplot(df["身体质量指数(BMI)"])
ax1.set_title("身体质量指数_箱型图")

ax2 = fig.add_subplot(122)
ax2.boxplot(df["血压"])
ax2.set_title("血压_箱型图")
plt.show()

In [ ]:
# 2. 直方图（年龄分布） + 饼图（性别分布）
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# 年龄直方图
ax1.bar(age_count.index.astype(str), age_count.values, color='skyblue', edgecolor='black')
ax1.set_title('年龄直方图')
ax1.set_xlabel('年龄段')
ax1.set_ylabel('人数')

# 性别饼图
# 饼图可以清晰反映部分与整体的比例关系，autopct 用于显示百分比
gender_count.plot(kind='pie', ax=ax2, autopct='%.1f%%', colors=['lightblue', 'lightpink'], startangle=90)
ax2.set_title('性别比例_饼图')
ax2.set_ylabel('') # 去掉 y 轴标签让图更好看

plt.tight_layout()
plt.show()

In [ ]:
# 3. 折线图：探查特征与病期（标签 y）的关系
# 因为血压是 100 多，病期是几十到几百，量纲完全不同，放在一张图里没法看！
# 所以这里我们引入【数据标准化】，(x - mean) / std，让大家都在同一个起跑线上（均值为0，标准差为1），再画折线图看趋势。

def standardize_for_plot(x):
    return (x - x.mean()) / x.std()

# 为了图表清晰，我们只取前 100 个样本来画图
sample_size = 100
x_axis = range(sample_size)
y_target = standardize_for_plot(df['发展阶段指数(病期)'][:sample_size])

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

# 血压 vs 病期
axes[0].plot(x_axis, standardize_for_plot(df['血压'][:sample_size]), color='red', label='标准化的血压')
axes[0].plot(x_axis, y_target, color='darkgreen', linestyle='--', label='标准化的病期')
axes[0].set_title("血压 与 发展阶段指数 的变化趋势对比")
axes[0].legend()

# 总胆固醇 vs 病期
axes[1].plot(x_axis, standardize_for_plot(df['总胆固醇'][:sample_size]), color='red', label='标准化的总胆固醇')
axes[1].plot(x_axis, y_target, color='darkgreen', linestyle='--', label='标准化的病期')
axes[1].set_title("总胆固醇 与 发展阶段指数 的变化趋势对比")
axes[1].legend()

# 血糖 vs 病期
axes[2].plot(x_axis, standardize_for_plot(df['血糖'][:sample_size]), color='red', label='标准化的血糖')
axes[2].plot(x_axis, y_target, color='darkgreen', linestyle='--', label='标准化的病期')
axes[2].set_title("血糖 与 发展阶段指数 的变化趋势对比")
axes[2].legend()

plt.tight_layout()
plt.show()
# 观察上面的折线图，如果红线（特征）和绿虚线（标签）起伏规律很像，说明它们相关性很强！线性回归模型最喜欢这种特征。

<hr><hr>

#### 第四步：特征工程 与 降维 (Dimensionality Reduction)
<hr>

In [ ]:
# 刚才我们生成了一个临时的“年龄段”列用于画图，现在丢掉它，因为模型不需要这种汉字类别，它需要原本的数字。
df.drop(labels='年龄段', axis=1, inplace=True)

# 【降维核心概念】：如果有的特征对预测目标完全没帮助，甚至会捣乱，我们就该删掉它。
# 这叫做 特征选择（降维的一种方式）：不改变原有特征、不生成新特征，仅通过筛选重要特征减少建模维度。
# 我们用“皮尔逊相关系数(Correlation)”来看看每个特征和目标值的关系。

# 相关系数取值 [-1, 1]。绝对值越接近 1，线性相关越强；接近 0，说明毫无关系。正数是正相关，负数是负相关。
correlations = df.corr()["发展阶段指数(病期)"].sort_values(ascending=False)
print("各特征与病期（标签）的相关系数：\n")
print(correlations)

In [ ]:
# 结果分析：
# 我们发现，'性别' 这个特征的相关系数绝对值特别小（约为 0.043），
# 这意味着在这个数据集里，性别对于预测一年后的糖尿病病期几乎没有什么线性参考价值。
# 它是一个“冗余特征”。丢掉它不仅能减少计算量，有时候还能提升模型的表现（因为模型不会被噪音干扰）。

df_reduced = df.drop(columns=['性别'])
print(f"\n删除了冗余特征'性别'后，数据集维度从 {df.shape[1]} 列变为了 {df_reduced.shape[1]} 列。")

<hr>

#### 第五步：‘测试集’和‘训练-验证集’的分割
<hr>

In [ ]:
SEED = 42 # 留个种子，制造伪随机，保证你我每次运行切出来的数据一模一样

# X 是特征（去掉了性别之后的 9 个生理指标）
X = df_reduced.drop(columns=['发展阶段指数(病期)']) 
# y 是标签（我们要预测的具体病期数值）
y = df_reduced['发展阶段指数(病期)'] 

# 同样，我们把 20% 锁进保险箱作为期末考试（测试集 Test），80% 留着平时练习（训练/验证集 Train_Val）
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.2,       
    random_state=SEED
    # 注意：回归问题通常不需要 stratify（分层抽样），因为 y 是连续的数字，而不是分类的离散标签。
)

print('数据集大小：')
print(f'训练-验证集: X={X_train_val.shape}, y={y_train_val.shape}')
print(f'测试集:       X={X_test.shape}, y={y_test.shape}')

<hr><hr>

#### 第六步：标准化 + Pipeline + KFold 交叉验证
<hr>

In [ ]:
# 回顾一下 Pipeline（流水线）：
# 它是为了防止“数据泄露”。我们要确保 Standardization（标准化）计算平均值和标准差时，
# 绝对不能偷看 测试集 的数据。

def build_regression_pipeline():
    """构建 StandardScaler + LinearRegression Pipeline"""
    return Pipeline([
        ('scaler', StandardScaler()),   # Step 1: 标准化（消除量纲影响）
        ('lin_reg', SklearnLinearRegression()) # Step 2: 线性回归模型
    ])

print('回归 Pipeline 结构:')
print(build_regression_pipeline())

# 为什么线性回归也要标准化？
# 理论上，普通的线性回归对量纲不敏感（它会自动调整权重 w 的大小来抵消单位影响）。
# 但是，做标准化是一个极好的工程习惯！它可以：
# 1. 提升模型训练收敛速度（在底层使用梯度下降求解时）。
# 2. 让我们在训练后，能够直接对比不同特征的权重（w）大小，判断哪个特征更重要。如果不标准化，血压的 w 和胆固醇的 w 完全没法比。

In [ ]:
##############################################################
# 线性回归有像 KNN 里的 'k' 那样的超参数需要调吗？
# 答案是：最基础的线性回归（LinearRegression）没有超参数。
# 它只有一个原则：找到一条线，让所有点到这条线的误差平方和（均方误差 MSE）最小。算出来是多少就是多少。
#
# 那我们还需要 KFold 5折交叉验证 吗？
# 依然非常需要！
# 如果我们不调参，KFold 的作用就变成了【模型稳定性评估】。
# 我们把 80% 的平时练习卷分成 5 份，轮流用 4 份训练，1 份自测。
# 这样能得出一个 5次的平均分。这个平均分能告诉我们：“这个模型到底靠不靠谱？是不是在某些特定的数据切片上表现极差？”
# 如果 5 次分数的波动 (std) 很大，说明模型非常不稳定，可能泛化能力很差。
##############################################################

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
fold_r2_scores = []

print('开始 KFold 交叉验证评估...\n')

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train_val)):
    # 取出这一折的 训练数据 和 验证数据
    X_fold_train = X_train_val.iloc[train_idx]
    X_fold_val   = X_train_val.iloc[val_idx]
    y_fold_train = y_train_val.iloc[train_idx]
    y_fold_val   = y_train_val.iloc[val_idx]

    # 初始化全新的 pipeline
    pipeline = build_regression_pipeline()
    
    # 在 4 份数据上训练
    pipeline.fit(X_fold_train, y_fold_train)
    
    # 在 1 份验证数据上算分
    # 注意：回归模型的默认 score 是 R2 (R-squared) 决定系数，而不是 Accuracy 准确率。
    score = pipeline.score(X_fold_val, y_fold_val)
    fold_r2_scores.append(score)
    print(f'  第 {fold_idx+1} 折 拟合指标 R2 = {score:.4f}')

mean_r2 = np.mean(fold_r2_scores)
std_r2 = np.std(fold_r2_scores)
print(f'\n结论：5折交叉验证平均 R2 = {mean_r2:.4f} ± {std_r2:.4f}')

<hr><hr>

#### 第七步：用全部训练数据重新训练，在 test 集上做<b>最终评估</b>
<hr>

In [ ]:
##############################################################
# 什么是 R2 (R-squared) 决定系数？
# 在分类里，我们用 Accuracy（猜对了百分之几）来打分。
# 但是在回归里，你要预测的是一个具体的数值（比如病期 151）。如果你预测了 151.01，算不算对？当然算很好了。
# 所以回归不用“猜对/猜错”来评价，而是看“误差有多小”。
# 
# R2 的通俗解释：
# - 满分是 1.0（100%）。表示你的预测点完全落在那条拟合的直线上，完美无瑕。
# - 如果是 0.0，表示你的模型和“瞎猜”一样烂。（这里的“瞎猜”指的是无脑猜所有人的病期都是平均值）。
# - R2 越高，表示模型对数据的拟合效果越好，特征对目标值的解释能力越强。
##############################################################

# 用完整的 train_val 复习一遍知识
final_pipeline = build_regression_pipeline()
final_pipeline.fit(X_train_val, y_train_val)

# 去考期末考试（test 集）
y_pred = final_pipeline.predict(X_test)

# 计算最终分数
test_r2 = r2_score(y_test, y_pred)
print(f'最终模型在期末考试(test集)上的拟合指标 R2：{test_r2:.4f}  (也就是 {test_r2*100:.2f}%)')

# 补充另外两个常用的回归误差指标：
# MSE (均方误差)：预测值减去真实值，平方，再求平均。
# MAE (平均绝对误差)：预测值减去真实值，取绝对值，再求平均（相当于预测值平均偏离了真实的病期多少点）。
print(f'MSE (均方误差): {mean_squared_error(y_test, y_pred):.2f}')
print(f'MAE (平均绝对误差): {mean_absolute_error(y_test, y_pred):.2f}')

In [ ]:
# 基线模型（Baseline）对比
# "什么都不学的傻模型"来了！
# 在回归里，最傻的策略是什么？就是不管病人长啥样，我都预测所有人的病期是“这批病人的平均病期”。
# 我们用 DummyRegressor 来模拟这个傻策略。

baseline = Pipeline([
    ('scaler', StandardScaler()),
    ('dummy', DummyRegressor(strategy='mean'))
])
baseline.fit(X_train_val, y_train_val)
baseline_r2 = baseline.score(X_test, y_test)
print(f'\n基线模型（盲猜平均值）在 test 集上的 R2：{baseline_r2:.4f}')
print(f'线性回归模型 提升了：{test_r2 - baseline_r2:+.4f}')
# 你会发现盲猜平均值的 R2 恰好非常接近 0。

<hr><hr>

#### 第八步：【拓展提高】从零推导一个线性回归模型
<hr>


刚才我们用 `sklearn`，几乎只用一句代码就训练好了模型。  
很多人在这个时候会产生一种感觉：**过程太快了，好像不知道机器到底是怎么“学习”的。**

其实 `sklearn` 的底层，本质上是直接利用数学公式求解（解析解）。  
但是，当数据规模达到千万级，或者模型变成拥有几十亿甚至上百亿参数的大语言模型（LLM）时，很多问题就已经**无法通过解析公式直接解出来了**。

因此，这里我们将彻底抛开 `sklearn` 的封装，只使用 **最基础的 Python 与数学逻辑**，手动实现一个完整的模型训练过程。

这其实就是 **深度学习框架（如 PyTorch / TensorFlow）底层思想的缩影**。

<br>

在正式写代码之前，需要先澄清一个**几乎所有初学者都会遇到的认知误区**。

很多人在听到：

> “我们要对 $y = wx + b$ 求导，然后用它指导参数更新。”

脑子会瞬间卡住：

> 一条直线的斜率不是固定的吗？  
> 对它求导有什么意义？  
> 它又是如何指导参数更新的？

请牢牢记住下面这句话：

> **梯度下降从来不是在 $y = wx + b$ 这张图上进行的。  
> 梯度下降发生在 Loss（误差）与参数之间构成的空间里。**

我们面对的是两个完全不同的世界：

| 世界 | 横轴 | 纵轴 | 含义 |
|---|---|---|---|
| **现实世界** | 输入 $x$ | 预测 $y$ | 数据与模型的关系 |
| **误差世界** | 参数 $w$ | Loss | 参数好坏的评价 |

换句话说：

- $y = wx + b$ 描述的是 **数据空间**
- 梯度下降发生在 **参数空间**

---

# 第一节：极简一维世界 —— “闭着眼睛下山”

为了让概念尽可能直观，我们先把问题简化到极致。

暂时去掉截距 $b$，只考虑：

$$
y_{预测} = w \times x
$$

假设只有 **一个数据点**

```
x = 2
y真实 = 6
```

我们的目标是找到合适的 $w$。

人类一眼就能看出来：

$$
w = 3
$$

因为

$$
3 \times 2 = 6
$$

但是计算机并没有这种直觉，它只能**从随机猜测开始逐步修正**。

---

## 1. 随机猜测并计算 Loss

假设模型一开始猜：

```
w = 1
```

那么预测值：

$$
y_{预测} = 1 \times 2 = 2
$$

真实值：

$$
y_{真实} = 6
$$

误差：

$$
2 - 6 = -4
$$

我们使用 **平方误差（MSE）**：

$$
Loss = (y_{预测} - y_{真实})^2
$$

因此：

$$
Loss = (2 - 6)^2 = 16
$$

现在请想象一张新的图：

- 横轴：$w$
- 纵轴：Loss

当 $w=1$ 时，Loss=16。

由于公式中存在平方项，这张图的形状一定是：

> **开口向上的抛物线**

而 Loss 最低的位置（锅底）就是：

```
w = 3
```

---

## 2. 用导数判断方向

现在模型站在：

```
w = 1
Loss = 16
```

它不知道锅底在哪。

它唯一能做的事情就是：

> **测量脚下的坡度**

也就是：

> **对 Loss 关于 w 求导**

Loss：

$$
Loss = (w \times 2 - 6)^2
$$

使用链式法则：

梯度：

$$
\frac{dLoss}{dw} = 2(w \times 2 - 6) \times 2
$$

把 $w=1$ 代入：

$$
2(1\times2 - 6)\times2
$$

得到：

```
-16
```

梯度为负数意味着：

> 当前坡度是 **左高右低**

因此如果 **向右移动（增大 w）**  
Loss 就会下降。

---

## 3. 更新参数

接下来需要一个控制步长的参数：

**学习率（Learning Rate）**

假设：

```
LR = 0.1
```

参数更新公式：

$$
w_{new} = w_{old} - LR \times gradient
$$

代入：

$$
w_{new} = 1 - 0.1 \times (-16)
$$

得到：

```
w = 2.6
```

仅仅一次更新：

```
1 → 2.6
```

新的预测：

```
y = 2.6 × 2 = 5.2
```

已经非常接近真实值：

```
6
```

这就是 **梯度下降的基本思想**：

> 在 Loss 构成的“地形”中，  
> 通过测量坡度，一步步走向最低点。

---

# 第二节：进入三维世界

现在我们把之前暂时去掉的 **截距 $b$** 加回来。

模型变成：

$$
y_{预测} = wx + b
$$

此时我们需要学习 **两个参数**

```
w
b
```

对应的 Loss 空间就从二维变成了三维：

```
Loss(w, b)
```

可以想象成一片真实的山脉：

| 维度 | 含义 |
|---|---|
| 经度 | w |
| 纬度 | b |
| 海拔 | Loss |

我们的目标：

> 找到 **最低的盆地**

计算机在这片地形中移动时，每一步都会计算：

```
∂Loss/∂w
∂Loss/∂b
```

这两个梯度分别告诉模型：

- w 方向坡度
- b 方向坡度

然后决定下一步往哪里走。

---

# 第三节：复杂地形与优化器

真实问题中的 Loss landscape  
远远没有抛物线那么简单。

它更像：

- 山脊
- 峭壁
- 深坑
- 平台

因此我们需要不同的 **优化器（Optimizer）**。

---

## 1. 传统 SGD

最原始的方法是：

**随机梯度下降（SGD）**

更新规则非常直接：

```
step = LR × gradient
```

如果梯度突然非常大，例如：

```
gradient = 10000
```

那么更新就会变成：

```
step = 0.1 × 10000 = 1000
```

参数会直接跳到非常远的位置。

这就可能导致：

```
数值溢出
Loss = NaN
```

这种现象叫做：

> **梯度爆炸（Gradient Explosion）**

常见解决方法：

#### 1. mini-batch

例如：
```
batch_size = 16
```
每次梯度是 **16 个样本平均值**
可以减少单个异常点的影响。

#### 2. 梯度裁剪

限制更新幅度：
```
if step > threshold:
    step = threshold
```
防止参数一步跳出合理范围。

---

## 2. Adam 优化器

Adam 是目前深度学习中最常用的优化器之一。

它的核心思想可以理解为：

> **利用历史梯度信息来调整当前步长**

主要包含两个记忆机制。

---

### 动量（Momentum）

记录梯度的方向趋势。

例如最近几步梯度为：

```
-10
10
-10
12
```

相互抵消后整体趋势很小。

而如果梯度持续：

```
20
30
30
```

那么下一步通常会继续朝这个方向移动。

---

### 方差估计（Variance）

记录梯度的平方：

```
-10 → 100
10 → 100
20 → 400
```

它作为 **分母项** 出现在更新公式中。

梯度波动越大：

```
分母越大
更新越小
```

可以理解为一种 **自动刹车机制**。

---

### EMA（指数移动平均）

Adam 不会无限记忆历史梯度。

旧梯度会按指数衰减：

$$
0.9^n
$$

这意味着：

- 最近梯度权重更高
- 很久以前的信息会逐渐被遗忘

---

### 偏差修正（Bias Correction）

训练初期历史信息较少。

Adam 会通过除以一个系数来进行修正，使得：

- 早期仍然可以大步探索
- 后期逐渐稳定

---


**以下是 adam的公式：**
![image.png](attachment:9dc35253-f04b-4d10-8e43-3a39d54df4b1.png)


In [4]:
import numpy as np
import time
from typing import List, Tuple, Dict, Any

class LinearRegression:
    def __init__(self):
        self.w = np.random.randn()
        self.b = np.random.randn()
        self.learning_rate = 0.001
        self.batch_size = 16
        self.epochs = 10001
        self.workers = 8
        self.optimizer = "Adam"
        self.loss_type = "MSE"
        self.loss = float("inf")
        self.loss_history = []
        self.sum_momentum_w = 0
        self.sum_variance_w = 0
        self.sum_variance_b = 0
        self.sum_momentum_b = 0

    def forward(self, X, w, b) -> List:
        # 计算预测值
        y_pred = []
        for x in X:
            y_pred.append(x * w + b)
        return y_pred

    def compute_loss(self, y_pred, y_true):
        if self.loss_type.lower() == "mse":
            loss = []
            for i in range(len(y_pred)):
                loss.append((y_pred[i] - y_true[i]) ** 2)
            return np.mean(loss)
        else:
            raise KeyboardInterrupt("Sorry , i didnt learn any other loss calculation method yet")

    def compute_gradients_w(self, X, y_pred, y_true):
        if self.loss_type.lower() == "mse":
            gradients = []
            for i in range(len(X)):
                gradients.append(2 * (y_pred[i] - y_true[i]) * X[i])
            return np.mean(gradients)
        else:
            raise KeyboardInterrupt("Sorry , i didnt learn any other loss calculation method yet")

    def compute_gradients_b(self, X, y_pred, y_true):
        if self.loss_type.lower() == "mse":
            gradients = []
            for i in range(len(X)):
                gradients.append(2 * (y_pred[i] - y_true[i]))
            return np.mean(gradients)
        else:
            raise KeyboardInterrupt("Sorry , i didnt learn any other loss calculation method yet")

    def Adam(self, EPOCH, GRADIENT_W, GRADIENT_B, beta1=0.9, beta2=0.99):
        """
        This is the adam optimizer wrote manually by myself, with the reference of functions taught by Gemini3.1pro(LLM).
        1. momentum (store the history of the vectors of the gradients, so that some 震荡 may 抵消， some 同一方向上的奔跑 may 放大)
        2. variance (control the speed of the 奔跑， store the 平方值 of the gradients, like the distance already ran. if too fast/ distance too much, then will be punished/ran slower);
        其实 w 和 b 的 gradients 应该放在同一个 **向量vector** 里的, 无奈我 numpy 不精, 所以把 w 和 b 都拆开存在variables里面
        """
        self.sum_momentum_w = (self.sum_momentum_w * beta1) + ((1-beta1)*GRADIENT_W)
        self.sum_momentum_b = (self.sum_momentum_b * beta1) + ((1-beta1)*GRADIENT_B)
        self.sum_variance_w = (self.sum_variance_w * beta2) + ((1-beta2)*(GRADIENT_W**2))
        self.sum_variance_b = (self.sum_variance_b * beta2) + ((1-beta2)*(GRADIENT_B**2))

        momentum_w_after_bias_correction = self.sum_momentum_w / (1 - beta1**EPOCH)
        momentum_b_after_bias_correction = self.sum_momentum_b / (1 - beta1**EPOCH)
        variance_w_after_bias_correction = self.sum_variance_w / (1 - beta2**EPOCH)
        variance_b_after_bias_correction = self.sum_variance_b / (1 - beta2**EPOCH)

        return momentum_w_after_bias_correction, momentum_b_after_bias_correction, variance_w_after_bias_correction, variance_b_after_bias_correction

    def new_w_b(self, original_w, original_b, gradient_w, gradient_b, lr=None, epoch=None, epsilong=(10**-8)):
        """
        使用刚才算出来的参数更新我们的w和b
        """
        if lr == None:
            lr = self.learning_rate # if no input, use default value

        update_of_w = None
        update_of_b = None

        if (self.optimizer.lower() == 'adam' or self.optimizer.lower() == 'adamw') and (epoch!=None):
            mw, mb, vw, vb = self.Adam(epoch, gradient_w, gradient_b)
            update_of_w = mw / ((vw)**0.5 + epsilong)
            update_of_b = mb / ((vb)**0.5 + epsilong)
        else:
            update_of_w = gradient_w
            update_of_b = gradient_b

        return (original_w - lr * update_of_w), (original_b - lr * update_of_b)

    def train_w(self, x_true, y_true):
        lr = self.learning_rate
        w = self.w
        b = self.b
        epoch = self.epochs
        loss = self.loss
        
        print("\n开始炼丹.")
        for _ in range(1, epoch):
            y_predicted = self.forward(x_true, w, b)
            loss = self.compute_loss(y_predicted, y_true)
            gradient_w = self.compute_gradients_w(x_true, y_predicted, y_true)
            gradient_b = self.compute_gradients_b(x_true, y_predicted, y_true)
            w, b = self.new_w_b(original_w=w, original_b=b, gradient_w=gradient_w, gradient_b=gradient_b, epoch=_)
                
            if _ % 1000 == 0:
                print(f"  > epoch={_}/{epoch}, loss={loss:.4f}, 当前的函数: y = {w:.2f}x + {b:.2f}")
                self.loss_history.append(loss)
                
        self.w = w
        self.b = b
        return w, b
    

def main():
    x = np.array([1, 2, 3, 4, 5])
    y = np.array([2, 4, 6, 8, 10])
    training = LinearRegression()
    
    try:
        w, b = training.train_w(x, y)
    except KeyboardInterrupt:
        print("\n training interrupted, 使用当前参数")
        w, b = training.w, training.b  # 使用当前值
    
    return training  # 返回整个模型对象


if __name__ == "__main__":
    model = main()
    print(f"\n最终结果: 模型拟合出的函数是 y = {model.w:.4f}x + {model.b:.4f}")
    print("（这已经极其接近完美的 y = 2x + 0 ）")


开始炼丹.
  > epoch=1000/10001, loss=14.6360, 当前的函数: y = 0.52x + 1.24
  > epoch=2000/10001, loss=0.8756, 当前的函数: y = 1.34x + 1.97
  > epoch=3000/10001, loss=0.2081, 当前的函数: y = 1.71x + 1.07
  > epoch=4000/10001, loss=0.0077, 当前的函数: y = 1.94x + 0.20
  > epoch=5000/10001, loss=0.0000, 当前的函数: y = 2.00x + 0.00
  > epoch=6000/10001, loss=0.0000, 当前的函数: y = 2.00x + -0.00
  > epoch=7000/10001, loss=0.0000, 当前的函数: y = 2.00x + -0.00
  > epoch=8000/10001, loss=0.0000, 当前的函数: y = 2.00x + -0.00
  > epoch=9000/10001, loss=0.0000, 当前的函数: y = 2.00x + 0.00
  > epoch=10000/10001, loss=0.0000, 当前的函数: y = 2.00x + 0.00

最终结果: 模型拟合出的函数是 y = 2.0000x + 0.0000
（这已经极其接近完美的 y = 2x + 0 ）


In [ ]:
# 画个图看看它训练时是怎么变聪明的
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
# 因为你改成了 100001 个 Epoch，并且每 1000 次记录一次，我们据此画出 loss_history
plt.plot(range(1000, model.epochs, 1000), model.loss_history, marker='.', color='purple', alpha=0.6)
plt.title('误差(y)-轮次(x)图')
plt.xlabel('Epoch轮次')
plt.ylabel('Loss损失函数（预测值和正确答案平均差多远）')
plt.grid(alpha=0.3)
plt.show()

我这个里面其实简化了，真正的adam是加上了 **L2正则化的**。<br>
正则化，尤其是L1和L2，是通过惩罚过大的w（限制w的大小）来预防**过拟合**的
<br><hr>
#### 正则化（Regularization）

真实训练中通常会加入：
```
L1
L2
```
正则化。

核心思想：
> **惩罚过大的参数**

例如 L2：
$$
Loss = MSE + \lambda w^2
$$
> 这样如果 $w$ 变得很大，Loss 会增加，模型一看：如果我加大w去精准穿过所有点固然好，但是大w会被惩罚（loss损失变大），于是就尽可能在小w上找最优解

---

#### 过拟合的数学本质

初学者往往把过拟合理解为：
> 像考试前死记硬背。

这个类比描述了现象，但没有揭示数学本质。
真正的问题在于：
> **参数过大导致函数过度弯曲。** (尤其是你去一元三次四次五次十一次 函数里代值算看看，如果w很大，模型会变得张牙舞爪，极度扭曲，想要够到每一个数据点）

> 其实这是不对的，对于有些过于极端的数据点，该放手就放手，不要尝试在训练集上做得太过完美，要学到一些真正有模式可循、带到新数据上也有用的规律

> 不止极端的数据点不需要被穿过，或者是本来两个相反方向的数据点，你在它们中间放条线就已经是最优解了，你偏偏去绕，强行让两个数据点都精确被线穿过，然后模型变得很复杂

如果尽力拟合所有数据点。结果是：
- 训练集误差很小
- 在没见过的数据上预测极差，达不到 在特定环境下预测新值的作用
  
正则化的作用正是：
> **限制参数规模，使函数保持平滑。**

> 注：*大w(权重)一般更容易发生过拟合，不代表小w(权重)完全不会发生过拟合，请理性看待。
---

#### 课后思考

值得深入思考的拓展问题：
> 1. 在Loss-w图像中，我们希望最后的最优解是一个碗底/谷底，而不是一个尖锐的深井，为什么？

> 2. 如果 $w$ 从 **0 附近开始优化**，  
> 而小 $w$ 区域往往更容易得到平滑解，  
> 为什么模型仍然可能最终落入 **尖锐深井**？

尤其是在使用 **Adam 优化器** 时，一旦进入某个深井，就很难再离开。
这个问题涉及：
- 优化路径
- 梯度统计
- 动量记忆
- Loss landscape 结构
可以先自己思考，再与 AI 或其他资料讨论。

#### 总结

| 阶段 | Sklearn 调包 | 你刚才手搓的（深度学习级） |
|------|------|------|
| **数学原理** | 解析解，直接背答案套公式。 | 梯度下降，一步一步试错和改正。 |
| **适用场景** | 传统机器学习，几万条数据。 | 深度学习、大语言模型、几百亿数据（完全通用）。 |
| **训练耗时** | 瞬间完成。 | 需要循环成千上万个 Epoch，炼丹时间长。 |
| **核心机制** | 数学矩阵求逆。 | 包含前向传播 (瞎猜)、误差计算 (挨打)、反向求导 (反思)、优化器更新参数 (改正)。 |
| **模型保存** | pickle (pkl) 文件 | 常用的 h5 / safetensors 格式，直接存取多维张量 (Tensors)。 |
<br><br><br>
> **关键点**：这可能是你人生中第一次真正“看清”人工智能大脑里发生的事情。从 `y = wx + b` 这个世界上最简单的公式，到包含几千亿个参数的 ChatGPT，本质上全都是：**猜答案 -> 算误差 -> 找下坡路(求导) -> 更新参数 (Adam优化)** 的枯燥循环。

<br>

> **学习建议**：你手搓的 `Adam` 和 `new_w_b` 函数非常精妙，它处理了所有的数学动量。如果你有兴趣，可以尝试把 `self.optimizer = "Adam"` 改成 `"SGD"`（直接进入传统的 else 分支），看看不用高级滑板鞋，光靠脚走（普通梯度下降），模型还能不能这么快找到答案！

<br>

<hr><hr>

## 实验二完成
<hr>

##### 此实验教材最近更新时间 2026年3月16日
<hr><hr>